### Importing useful libraries

In [1]:
! pip install datasets;
!pip install unicodedataplus;

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.0/763.0 kB 11.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for unicodedataplus: filename=unicodedataplus-16.0.0-cp311-cp311-linux_x86_64.whl size=643211 sha256=73d552ef25e347489e7549407e99b9e48531a921f1edd3fbe8a00851e5991fb1
  Stored in directory: /root/.cache/pip/wheels/3e/4d/f4/bbdb70ef91c05fa54fafe2d72dae5cbc4b577b7ed6ed1d8604
Successfully built unicodedataplus


In [30]:
import pandas as pd
import numpy as np
import string
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

model_name = "bert-base-multilingual-cased"

In [19]:
!unzip '/content/mbert-weigths.zip' -d 'mbert-weights'

Archive:  /content/mbert-weigths.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/mbert-weigths.zip or
        /content/mbert-weigths.zip.zip, and cannot find /content/mbert-weigths.zip.ZIP, period.


### Data preprocessing

Importing Dataset

In [48]:
df = pd.read_csv("train_submission.csv")
df.drop(columns=['Usage'],inplace=True)
df.dropna(inplace=True)

Encoding labels as numbers

In [49]:
label_encoder = LabelEncoder()
df["labels"] = label_encoder.fit_transform(df['Label'])
label_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}
df.head()

,Text,Label,labels
0,َ قَالَ النَّبِيُّ ص إِنِّي أَتَعَجَّبُ مِمَّن...,hau,119
1,Filmen forteller historien om Will Hunting en...,nob,247
2,An Arthrostylidium berryi in uska species han ...,wln,371
3,Kancunarí enemigosniyquichejta munacuychej al...,quh,282
4,Warmeqa ama yachachichunchu hermanospa tantaku...,quh,282


Tokenizing the texts

In [50]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['Text'], padding = "max_length",truncation = True, return_tensors='pt')

In [51]:
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(tokenize_function,batched=True)
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/190099 [00:00<?, ? examples/s]

Splitting train/test datasets

In [52]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

Classification using mBERT

In [53]:
num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
def scripture_guess(texts):
    #return scripture category for a batch of texts
    labels_guessed = []
    for text in texts:
        sample = text[:20]
        guess = []
        for c in sample:
            guess.append(unicodedataplus.script(c))

        labels_guessed.append(max(set(guess), key=guess.count))
    return labels_guessed

# List of Scriptures used only in one language
exclusive_scripture = {'Thaana': 'div', 'Gujarati': 'guj', 'Canadian_Aboriginal': 'iku', 'Kannada': 'kan', 'Khmer': 'khm', 'Hangul': 'kor', 'Lao': 'lao', 'Malayalam': 'mal', 'Gurmukhi': 'pan', 'Ol_Chiki': 'sat', 'Sinhala': 'sin', 'Tamil': 'tam', 'Thai': 'tha'}

Define Training Loop

In [55]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import unicodedataplus

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 10
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(val_dataset, batch_size=32)
loss_values = []
optimizer = Adam(model.parameters(), lr=2e-5)

for epoch in range(num_epochs):
    model.train()
    print(f"Epoch {epoch+1}/{num_epochs}")
    curr_loss_val = []
    for ite,batch in tqdm(enumerate(train_dataloader)):
        inputs = {key: val.to(device) for key, val in batch.items() if key in ['labels','input_ids','token_type_ids','attention_mask']}
        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        curr_loss_val.append(loss.item())
        if ite % 100 == 0:
            loss_values.append(np.mean(curr_loss_val))
            print(f"Loss after {ite} iterations: {np.mean(curr_loss_val)}")
            curr_loss_val = []
    model.save_pretrained(f"./model_epoch_{epoch+1}")
    tokenizer.save_pretrained(f"./model_epoch_{epoch+1}")  # Save the tokenizer as well

    model.eval()  # Set the model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():  # No need to track gradients during validation
        for batch in val_dataloader:
            scriptures = scripture_guess(batch['Text'])
            inputs = {key: val.to(device) for key, val in batch.items() if key in ['labels','input_ids', 'token_type_ids','attention_mask']}
            outputs = model(**inputs)
            logits = outputs.logits

            # Get the predicted class (index with the highest logit value)
            preds = torch.argmax(logits, dim=-1)
            script_preds = {i:exclusive_scripture[s] for i,s in enumerate(scriptures) if s in exclusive_scripture.keys()}
            for key,value in script_preds.items():
                preds[key] = label_encoder.transform([value])[0]

            all_preds.extend(preds.cpu().numpy())  # Move predictions to CPU for further use
            all_labels.extend(inputs["labels"].cpu().numpy())  # True labels

    # Calculate accuracy on the validation set
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"Validation Accuracy after Epoch {epoch+1}: {accuracy:.4f}")


Epoch 1/10


1it [00:01,  1.19s/it]

Loss after 0 iterations: 5.949946880340576


101it [01:54,  1.14s/it]

Loss after 100 iterations: 5.8625872564315795


...

Loss after 1300 iterations: 0.4267695562541485


1401it [26:32,  1.14s/it]

Loss after 1400 iterations: 0.4215404757857323


1485it [28:08,  1.14s/it]


KeyboardInterrupt: 

In [9]:
def scripture_guess(texts):
    #return scripture category for a batch of texts
    labels_guessed = []
    for text in texts:
        sample = text[:20]
        guess = []
        for c in sample:
            guess.append(unicodedataplus.script(c))

        labels_guessed.append(max(set(guess), key=guess.count))
    return labels_guessed

# List of Scriptures used only in one language
exclusive_scripture = {'Thaana': 'div', 'Gujarati': 'guj', 'Canadian_Aboriginal': 'iku', 'Kannada': 'kan', 'Khmer': 'khm', 'Hangul': 'kor', 'Lao': 'lao', 'Malayalam': 'mal', 'Gurmukhi': 'pan', 'Ol_Chiki': 'sat', 'Sinhala': 'sin', 'Tamil': 'tam', 'Thai': 'tha'}

100%|██████████| 485/485 [01:39<00:00,  4.87it/s]

Validation Accuracy after Epoch 1: 0.0023


In [ ]:
import shutil
shutil.make_archive('mbert-weigths', 'zip', 'model_epoch_6')

'/content/mbert-weigths.zip'

In [ ]:
from collections import defaultdict

def per_class_accuracy(predictions, true_labels):
    class_correct = defaultdict(int)  # Count of correct predictions per class
    class_total = defaultdict(int)    # Count of total samples per class

    for pred, true in zip(predictions, true_labels):
        class_total[label_mapping[true]] += 1  # Count total occurrences of this class
        if pred == true:
            class_correct[label_mapping[true]] += 1  # Count correct predictions for this class

    # Compute accuracy per class
    class_accuracy = {cls: class_correct[cls] / class_total[cls] for cls in class_total}

    return class_accuracy

per_class_accuracy(all_preds,all_labels)

{'deu': 1.0,
 'uzn': 0.6666666666666666,
 'mad': 0.38095238095238093,
 'grn': 0.8,
 'zai': 1.0,
 'lus': 0.9166666666666666,
 'bar': 0.8888888888888888,
 'rop': 1.0,
 'bos': 0.6,
 'cbk': 0.8333333333333334,
 'zul': 0.18181818181818182,
 'arg': 0.8,
 'tgk': 0.4523809523809524,
 'pcm': 0.6666666666666666,
 'quw': 1.0,
 'fry': 0.7142857142857143,
 'quc': 0.9090909090909091,
 'mhr': 1.0,
 'tsn': 0.46153846153846156,
 'sah': 0.9090909090909091,
 'ast': 1.0,
 'fij': 1.0,
 'umb': 0.9230769230769231,
 'bsb': 1.0,
 'fil': 0.4666666666666667,
 'yao': 1.0,
 'cab': 1.0,
 'sun': 0.16666666666666666,
 'tuk': 0.9523809523809523,
 'crh': 0.8888888888888888,
 'lug': 1.0,
 'iku': 0.6,
 'ita': 0.625,
 'knv': 1.0,
 'tyv': 0.9411764705882353,
 'ace': 1.0,
 'twi': 0.3333333333333333,
 'gsw': 0.0,
 'kur': 0.7083333333333334,
 'lua': 1.0,
 'sin': 0.3333333333333333,
 'lin': 0.8235294117647058,
 'gug': 1.0,
 'nde': 0.6666666666666666,
 'vec': 0.4,
 'aka': 0.4,
 'sag': 1.0,
 'bod': 0.75,
 'mgh': 0.9,
 'ahk': 1.0

In [12]:
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import unicodedataplus

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32)
val_dataloader = DataLoader(val_dataset, batch_size=16)

model.eval()  # Set the model to evaluation mode
all_preds = []
all_labels = []
epoch = 0

with torch.no_grad():  # No need to track gradients during validation
    for batch in tqdm(val_dataloader):
        scriptures = scripture_guess(batch['Text'])
        inputs = {key: val.to(device) for key, val in batch.items() if key in ['labels','input_ids', 'token_type_ids','attention_mask']}
        outputs = model(**inputs)
        logits = outputs.logits

        # Get the predicted class (index with the highest logit value)
        preds = torch.argmax(logits, dim=-1)
        script_preds = {i:exclusive_scripture[s] for i,s in enumerate(scriptures) if s in exclusive_scripture.keys()}
        for key,value in script_preds.items():
            preds[key] = label_encoder.transform([value])[0]

        all_preds.extend(preds.cpu().numpy())  # Move predictions to CPU for further use
        all_labels.extend(inputs["labels"].cpu().numpy())  # True labels

# Calculate accuracy on the validation set
accuracy = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy after Epoch {epoch+1}: {accuracy:.4f}")

100%|██████████| 243/243 [00:23<00:00, 10.30it/s]

Validation Accuracy after Epoch 1: 0.0328


Adding Vocabulary Prediction for rare language

In [12]:
df = pd.read_csv("train_submission.csv")

X = df['Text'].to_list()
y = df['Label'].to_list()
labels = df.groupby('Label').first().reset_index()['Label'].to_list()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [13]:
texts_dict = {}

for label in tqdm(labels):
    X_lang = [X_train[i] for i,x in enumerate(y_train) if x == label]
    texts_dict[label] = X_lang

100%|██████████| 389/389 [00:00<00:00, 582.53it/s]


In [14]:
import unicodedataplus

def scripture_guess(texts):
    labels_guessed = []
    for text in texts:
      sample = text[:20]
      guess = []
      for c in sample:
          guess.append(unicodedataplus.script(c))
      return max(set(guess), key=guess.count)

l2a_dict = {}
a2l_dict = {}

labels = label_encoder.classes_
for label in labels:
    guess = []
    for text in texts_dict[label]:
        sample = text[:20]
        for c in sample:
            guess.append(unicodedataplus.script(c))

    scripture = max(set(guess), key=guess.count)
    l2a_dict[label] = scripture
    if scripture not in a2l_dict.keys():
        a2l_dict[scripture] = []

    a2l_dict[scripture].append(label)

In [17]:
single_value_keys = {key:value[0] for key, value in a2l_dict.items() if len(value) == 1}

print(single_value_keys)

{'Thaana': 'div', 'Gujarati': 'guj', 'Canadian_Aboriginal': 'iku', 'Kannada': 'kan', 'Khmer': 'khm', 'Hangul': 'kor', 'Lao': 'lao', 'Malayalam': 'mal', 'Gurmukhi': 'pan', 'Ol_Chiki': 'sat', 'Sinhala': 'sin', 'Tamil': 'tam', 'Thai': 'tha'}


## Computing Predictions

In [46]:
test_df = pd.read_csv("test_without_labels.csv")
test_df.drop(columns=['ID','Usage'],inplace=True)

test_dataset = Dataset.from_pandas(test_df)
tokenized_dataset = test_dataset.map(tokenize_function,batched=True)
tokenized_dataset.set_format("torch")

val_dataloader = DataLoader(tokenized_dataset, batch_size=16)

model.eval()  # Set the model to evaluation mode
all_preds = []

with torch.no_grad():  # No need to track gradients during validation
    for batch in tqdm(val_dataloader):
        scriptures = scripture_guess(batch['Text'])
        inputs = {key: val.to(device) for key, val in batch.items() if key in ['labels','input_ids', 'token_type_ids','attention_mask']}
        outputs = model(**inputs)
        logits = outputs.logits

        # Get the predicted class (index with the highest logit value)
        preds = torch.argmax(logits, dim=-1)
        script_preds = {i:exclusive_scripture[s] for i,s in enumerate(scriptures) if s in exclusive_scripture.keys()}
        for key,value in script_preds.items():
            preds[key] = label_encoder.transform([value])[0]

        all_preds.extend(preds)  # Move predictions to CPU for further use

test_df['labels'] = all_preds


Map:   0%|          | 0/38827 [00:00<?, ? examples/s]

 23%|██▎       | 551/2427 [00:53<03:02, 10.28it/s]


KeyboardInterrupt: 